# TEXT TO IMAGE USING STABLE DIFFUSION 

Stable diffusion is a text to image latent diffusion model. Diffusion models are a type of generative model used in machine learning to create data like image or audio. Diffusion models work by gradually adding noise over several steps to a data (an image) until it is completely random noise. It then learns to reverse this process, consequently generating new samples. However, latent diffusion models are different from general diffusion models. Latent diffusion models operates in a compressed latent space while general diffusion models directly operate on pixel space. This consumes more memory, making the reverse denoising process slow. Stable diffusion is a latent diffusion model. 

## Lab Description:

In this lab, participants will explore Stable Diffusion, a powerful AI model for generating images from text prompts. They will load a StableDiffusionPipeline, generate images using random seeds to observe variations, and experiment with generating multiple images with the same prompt.

## Lab Objectives

After completing the lab, the participants would be able to :

- Load and Utilize the Stable Diffusion Pipeline to generate images from text prompts.

- Understand and Apply Random Seeds to observe variations in image outputs and control reproducibility.

- Generate and Arrange Multiple Images in a Grid for better visualization and comparison.

- Modify Prompts and Model Parameters to explore their impact on image generation and fine-tune results.

## Stable Diffusion Flow

<div style="text-align: center;">
    <img src="flow.png" alt="Description" width="720" height="480">
</div>

## The`StableDiffusionPipeline`

`StableDiffusionPipeline` is an inference pipeline used to generate images from text. 

First, we load the pre-trained weights of all components of the model. We use stable diffusion version 1.4 for this lab. Other version can also be used. We pass the model id and `torch_dtype` to the `from_pretrained` method. 

`torch_dtype=torch.float16` tells the model to expect weights in float16 (16-bit) precision. While this reduces precision, it also reduces memory usage and computation time. Removing the argument will load the weights in default float32 (32-bit) precision (which will be more accurate but might require more memory).

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    torch_dtype=torch.float16
)
pipe = pipe.to("cuda")

We are now all set to generate images. 


In [ ]:
prompt = "a beautiful landscape of mountains at sunset"
image = pipe(prompt).images[0]


We can either display the image like :

In [ ]:
image

Or we can save the image like this :

In [ ]:
image.save(f"gen_image.png")

## Usage of Random Seed

Let us try to execute the same image generation cell once again.

In [ ]:
prompt = "a beautiful landscape of mountains at sunset"
image = pipe(prompt).images[0]

image

We can see that we got an image different from the first one. 

What if we want to generate the same image everytime we execute image generation ? 

This is where random seed comes into play. 

You can set a random seed using `torch.generator`. `torch.generator` is a python class that provides mechanisms to create and manage random number generators or RNGs. 

The `.manual_seed()` method helps to set a seed for the generator. A seed is the starting point of the sequence of random numbers generated by the random number generator (RNG). This means that if you set a particular seed, whenever you use that seed value, the same sequence gets reproduced. This ensures reproducibility. Like in our case, we can set a manual seed and get the same image generated everytime.  


Try executing the cell multiple times to see if the generated image changes. 

In [ ]:
generator = torch.Generator("cuda").manual_seed(1024)

image = pipe(prompt, generator=generator).images[0]

image

## The `num_inference_steps` argument

`num_inference_steps` is the number of discrete steps in the reverse diffusion process. It is the number of iterations the model goes through, to finally denoise the image. Results are better when you use more number of steps. 

Try changing the number of inference steps to see what changes happen to the generated image. 

In [ ]:
generator = torch.Generator("cuda").manual_seed(1024)

image = pipe(prompt, num_inference_steps=15, generator=generator).images[0]

image

## The `guidance_scale` argument

### Classifier free guidance

A Diffusion model can generate image in two ways:

It can generate an image without any guidance. That is, it generates image purely based on the learning from the training data. 

Or, it can generate an image conditioned on a text prompt.

CLassifier free guidance takes into account both these outputs. 

The guidance scale determines how strongly the model favors the conditioned output over the unconditioned output based on this formula:

$$
\text{guided\_output} = \text{unconditioned\_output} + \text{guidance\_scale} \times (\text{conditioned\_output} - \text{unconditioned\_output})
$$

Let us see how to use the `guidance_scale` argument

In [ ]:
# Set the guidance scale
guidance_scale = 7.5  # Common values range from 7 to 12

# Generate an image using the pipeline with the guidance scale
image = pipe(prompt, guidance_scale=guidance_scale).images[0]

# Display the image
image


If you use low guiadance values (1-5), The model will generate more creative image, but it might not follow the text prompt accurately.

If you use high guidance values (10-20+), the model strictly follow the prompt and the generated image will look very artificial and less meaningful.

The values between 7 and 12 are the best to keep balance between diversity and adherence to the prompt. 

## Generation of multiple images based on the same prompt

We can generate `n` images based on a prompt by simply repeating the prompt `n` times.

In [ ]:
num_images = 9
prompt = ["a beautiful landscape of mountains at sunset"] * num_images

images = pipe(prompt).images

images

We got a list with 3 image objects. We can define a simple grid function to display all these images together.

In [ ]:
from PIL import Image

def generate_grid(imgs, rows, cols):
    assert len(imgs) == rows*cols

    w, h = imgs[0].size
    grid = Image.new('RGB', size=(cols*w, rows*w))
    grid_w, grid_h = grid.size
    
    for i, img in enumerate(imgs):
        grid.paste(img, box=(i%cols*w, i//rows*h))
    return grid    

We can define the number of rows and columns for the grid. Since we have 9 images, we can define `num_rows = 3`, `num_cols = 3`.

In [ ]:
num_rows = 3
num_cols = 3

img_grid = generate_grid(images, num_rows, num_cols)

img_grid

We generated 9 different images with the same prompt and displayed it on a grid.

<div style="text-align: left;">
    <img src="logo.png" alt="Description" width="150" height="100">
</div>